# Project Top

# Goal
Your goal will be to determine how the MLB rule changes in 2023 affected the value of infield defense.  This could be from a player evaluation, overall run prevention, or some other perspective. The data set will include SIS proprietary defensive measures, player location, and hit location. Data dictionary and references will be provided. The data can be merged with any free publicly available data set. Use of AI is permitted. AI sources should be cited. Students are responsible for understanding their output and must be able to answer questions about methodology.

## Dataset
Dataset: 2022-2023 selected infield play by play including SIS created metrics

## Variables  
Season - season the game was played.  
    
GameId - SIS's internal game ID. 
       
Inning - which inning the play is from.  
       
InningHalf - which half-inning the play is from (0 is the top half, 1 is the bottom half).     
    
AtBatIndex - which at-bat within the inning the play is from.     
    
OutsBefore - how many outs had been recorded earlier in the inning.     
    
RunnerOn1B - was there a runner on first base.     
    
RunnerOn2B - was there a runner on second base.   
    
RunnerOn3B - was there a runner on third base.     
     
BatSide - batter handedness (left or right).     
     
Pos - fielder position.     
    
FielderAngle - angle of the fielder relative to home plate, in degrees. negative numbers are to the left, positive numbers are to the right from the catcher's perspective.     
     
FielderDepth - depth of the fielder relative to home plate, in feet.   
       
BallInPlayType - ball in play type (grounder or liner).     
     
BallInPlayAngle - angle of the ball in play relative to home plate, in degrees. negative numbers are to the left, positive numbers are to the right from the catcher's perspective.    
       
BallInPlayDistance - distance that the ball traveled. on a groundball, it's where it was fielded. on a live drive, it's where the ball first contacted something.      
      
BattedBallVelocityGroup - categorization for how hard the ball was hit, where 1 is soft, 2 is medium, and 3 is hard. this correpsonds to the smae categories on FanGraphs' stats pages.     
       
ShiftType - defensive shift type: Full (3 fielders to 1 side), Partial (not Full, but substaintially different from typical), None (no meaningful shift), and Other (a catch-all for situational shifts, e.g. "Infield In").      
     
BatterSpeedGroup - SIS's internal speed score for the batter, on a scale of 1 (slow) to 5 (fast). 
           
DistanceToBaseGroup - for the fielding player, roughly how far was he from the most likely base he should throw to when he fielded the ball.   
        
TimeForThrowGroup - for the fielding player, roughly how much time should he have to complete the play, given the speed of the runner and the timing of the batted ball. 
          
ExpOutRate_NoPositioning - expected out rate (EOR) for this player given historical precedence, knowing only the characteristics of the batted ball.   
        
ExpOutRate_NoPositioning_Team - EOR for the team given historical precedence, knowing only the characteristics of the batted ball.      
      
ExpOutRate_WithPositioning - EOR for this player given historical precedence, knowing only the characteristics of the batted ball and the alignment of the fielders.       
     
ExpOutRate_WithPositioning_Team - EOR for the team given historical precedence, knowing only the characteristics of the batted ball and the alignment of the fielders.    
         
ExpOutRate_WhenFielded - EOR for this player after the ball is fielded, given how far the throw is and how much time he has to complete the play.      
     
ActualOut - binary "was the out recorded by this player", where 1 is a success.      



# Packages

In [ ]:
%pip install pandas 
%pip install numpy
%pip install seaborn
%pip install matplotlib
%pip install plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Dataset Conversion and Initial Checks

In [3]:
raw_data = pd.read_csv("OSU Hackathon 2026 data.csv")
raw_data.head()


,Season,GameId,Inning,InningHalf,AtBatIndex,OutsBefore,RunnerOn1B,RunnerOn2B,RunnerOn3B,BatSide,...,ShiftType,BatterSpeedGroup,DistanceToBaseGroup,TimeForThrowGroup,ExpOutRate_NoPositioning,ExpOutRate_NoPositioning_Team,ExpOutRate_WithPositioning,ExpOutRate_WithPositioning_Team,ExpOutRate_WhenFielded,ActualOut
0,2022,20220001,1,0,5,0,0,1,0,L,...,Partial,4,NaN,NaN,0.039,0.569,0.000,0.000,NaN,0
1,2022,20220001,1,0,5,0,0,1,0,L,...,Partial,4,NaN,NaN,0.461,0.569,0.000,0.000,NaN,0
2,2022,20220001,1,0,5,0,0,1,0,L,...,Partial,4,NaN,NaN,0.069,0.569,0.000,0.000,NaN,0
3,2022,20220001,1,1,1,0,0,0,0,R,...,Full,1,80.0,2500.0,0.879,0.879,0.549,0.549,0.942,1
4,2022,20220001,2,0,3,1,1,0,0,R,...,NaN,2,70.0,3200.0,0.879,0.879,0.896,0.896,0.884,1


In [4]:
print("Dataset Info:")
raw_data.info()

print('\n' + '~'*80)
print("Summary Statistics:")
display(raw_data.describe())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 188021 entries, 0 to 188020
Data columns (total 27 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Season                           188021 non-null  int64  
 1   GameId                           188021 non-null  int64  
 2   Inning                           188021 non-null  int64  
 3   InningHalf                       188021 non-null  int64  
 4   AtBatIndex                       188021 non-null  int64  
 5   OutsBefore                       188021 non-null  int64  
 6   RunnerOn1B                       188021 non-null  int64  
 7   RunnerOn2B                       188021 non-null  int64  
 8   RunnerOn3B                       188021 non-null  int64  
 9   BatSide                          188021 non-null  str    
 10  Pos                              188021 non-null  str    
 11  FielderAngle                     188012 non-null  float64
 12 

,Season,GameId,Inning,InningHalf,AtBatIndex,OutsBefore,RunnerOn1B,RunnerOn2B,RunnerOn3B,FielderAngle,...,BattedBallVelocityGroup,BatterSpeedGroup,DistanceToBaseGroup,TimeForThrowGroup,ExpOutRate_NoPositioning,ExpOutRate_NoPositioning_Team,ExpOutRate_WithPositioning,ExpOutRate_WithPositioning_Team,ExpOutRate_WhenFielded,ActualOut
count,188021.000000,1.880210e+05,188021.000000,188021.000000,188021.000000,188021.000000,188021.000000,188021.000000,188021.000000,188012.000000,...,188021.000000,188021.000000,82198.000000,82198.000000,188021.000000,188021.000000,188021.000000,188021.000000,82198.000000,188021.000000
mean,2022.484105,2.022605e+07,4.919711,0.488531,2.824934,0.970711,0.291202,0.195632,0.091506,-4.133172,...,2.128081,3.266731,73.063943,2523.326602,0.403716,0.701171,0.405651,0.703928,0.927885,0.405928
std,0.499749,5.054642e+03,2.562303,0.499870,1.635048,0.815294,0.454317,0.396688,0.288328,24.189810,...,0.636678,1.101160,37.751947,484.053954,0.354614,0.264209,0.402329,0.321641,0.105582,0.491072
min,2022.000000,2.022000e+07,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,-45.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.059000,0.000000
25%,2022.000000,2.022115e+07,3.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,-23.000000,...,2.000000,2.000000,35.000000,2200.000000,0.045000,0.522000,0.000000,0.528000,0.908000,0.000000
50%,2022.000000,2.022235e+07,5.000000,0.000000,3.000000,1.000000,0.000000,0.000000,0.000000,-5.000000,...,2.000000,3.000000,70.000000,2600.000000,0.317000,0.791000,0.287000,0.834000,0.958000,0.000000
75%,2023.000000,2.023119e+07,7.000000,1.000000,4.000000,2.000000,1.000000,0.000000,0.000000,18.000000,...,3.000000,4.000000,110.000000,2900.000000,0.774000,0.915000,0.849000,0.949000,1.000000,1.000000
max,2023.000000,2.023243e+07,15.000000,1.000000,14.000000,2.000000,1.000000,1.000000,1.000000,45.000000,...,3.000000,5.000000,320.000000,3900.000000,1.000000,2.224000,1.000000,1.813000,1.000000,1.000000


In [5]:
# How do the SIS metrics differ with the introduction of rule change in 2023

group_cols = ['Season', 'ShiftType']
val_cols = ['ExpOutRate_NoPositioning_Team','ExpOutRate_WithPositioning_Team','ExpOutRate_WhenFielded']
expoutrate = raw_data.groupby(group_cols)[val_cols].mean()
expoutrate

ExpOutRate_NoPositioning_Team  \
Season ShiftType                                  
2022   Full                            0.717971   
       Other                           0.684687   
       Partial                         0.696886   
2023   Other                           0.693873   
       Partial                         0.697261   

                  ExpOutRate_WithPositioning_Team  ExpOutRate_WhenFielded  
Season ShiftType                                                           
2022   Full                              0.750959                0.935680  
       Other                             0.678963                0.933577  
       Partial                           0.683372                0.925963  
2023   Other                             0.688446                0.941401  
       Partial                           0.693702                0.933885

In [6]:
group_cols = ['Season', 'ShiftType']
val_cols = ['FielderAngle','FielderDepth','BallInPlayDistance', 'TimeForThrowGroup']
fielding = raw_data.groupby(group_cols)[val_cols].mean()
fielding

FielderAngle  FielderDepth  BallInPlayDistance  \
Season ShiftType                                                   
2022   Full           2.145679    138.930649          143.350433   
       Other         -2.304726    118.234655          159.898968   
       Partial       -6.705508    134.954603          145.285228   
2023   Other         -2.329236    113.216996          156.128808   
       Partial       -3.147684    134.531250          146.409355   

                  TimeForThrowGroup  
Season ShiftType                     
2022   Full             2517.161310  
       Other            2647.641073  
       Partial          2498.564014  
2023   Other            2722.191011  
       Partial          2538.233535

In [8]:
group_cols = ['ShiftType']
val_cols = ["ExpOutRate_NoPositioning", "ExpOutRate_WithPositioning"]
individual = raw_data.groupby(group_cols)[val_cols].mean()
individual

,ExpOutRate_NoPositioning,ExpOutRate_WithPositioning
ShiftType,,
Full,0.298164,0.306236
Other,0.595470,0.591713
Partial,0.330273,0.330691


# Initial Configurations

In [14]:
raw_data['Pos'].unique()

<ArrowStringArray>
['3B', 'SS', '2B', '1B']
Length: 4, dtype: str

In [16]:
# Folder for visuals
OUTPUT_DIR = "./figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# Positions considered corner infielders
CORNER_POSITIONS = ['1B', '3B'] # 1B = 3, 3B = 5 in standard numeric labels
 
# Seaborn theme
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)
YEAR_PALETTE = {2022: "#4878CF", 2023: "#E8735A"}